# 🎤 Seed-VC 歌声音色转换 | Song Voice Timbre Conversion

> **Zero-shot singing voice conversion** — no training required.  
> **零样本歌声音色转换** — 无需训练，开箱即用。

---

## What this notebook does / 本 Notebook 的功能

| | 中文 | English |
|---|---|---|
| **输入 Input** | `reference.wav` — 20~30 秒目标音色参考人声 | 20–30s reference voice (target timbre) |
| **输入 Input** | `source.wav` — 完整原曲人声（3~5 分钟） | Full-length source vocal (3–5 min) |
| **输出 Output** | 保留原曲旋律节奏，音色替换为 reference 的声音 | Same melody & timing, voice replaced with reference timbre |

---

## ⚙️ Before you start — Kaggle setup / 运行前必看

1. **GPU** — Right panel → **Accelerator** → select `GPU T4 x2` or `T4`  
   右侧面板 → **Accelerator** → 选择 `GPU T4 x2` 或 `T4`

2. **Upload your audio as a Kaggle Dataset** and add it via the **Input** panel (left sidebar).  
   将你的音频上传为 Kaggle Dataset，然后在左侧 Input 面板中挂载。

3. **Edit `REF_PATH` and `SRC_PATH`** in Cell 1 to match your file paths.  
   在 Cell 1 中修改 `REF_PATH` 和 `SRC_PATH` 为你的实际文件路径。

---

## 📋 Required files / 需要准备的文件

| 文件 | 要求 | Requirement |
|------|------|-------------|
| Reference voice (参考人声) | 20~30 秒，纯人声，已拆轨 | 20–30s, clean vocal, no BGM |
| Source vocal (原始人声) | 完整歌曲人声，纯人声，已拆轨 | Full song vocal, clean, no BGM |

> 💡 Use [UVR5](https://github.com/Anjok07/ultimatevocalremovergui) or [Demucs](https://github.com/facebookresearch/demucs) to separate vocals from music before using this notebook.  
> 💡 使用 UVR5 或 Demucs 提前拆轨，去除背景音乐，否则转换效果会很差。

---

## 🗺️ Workflow overview / 流程概览

```
Cell 1  →  配置路径和参数     Configure paths & parameters
Cell 2  →  检查 GPU 环境      Verify GPU is available
Cell 3  →  验证音频文件存在    Validate input audio files
Cell 4  →  安装 Seed-VC       Install Seed-VC & dependencies
Cell 4.5→  修复依赖冲突        Fix protobuf conflict (Kaggle only)
Cell 5  →  复制音频到工作目录  Copy audio to working directory
Cell 6  →  20秒片段测试推理    Test inference on 20s preview
Cell 7  →  完整歌曲推理        Full song inference
Cell 8  →  打包下载结果        Package & download results

Cells 6.1~6.3  →  诊断工具（仅出错时使用）  Diagnostic tools (only if issues arise)
```

## Cell 1 — Global Configuration / 全局配置

> **⚠️ This is the only cell you need to edit before running. / 这是运行前唯一需要修改的 Cell。**

Change `REF_PATH` and `SRC_PATH` to your actual Kaggle Dataset paths.  
将 `REF_PATH` 和 `SRC_PATH` 改为你在 Kaggle 上挂载的实际路径。

**How to find your path / 如何找到路径：**  
Left sidebar → Input → hover over your file → copy the path shown.  
左侧 Input 面板 → 鼠标悬停文件 → 复制显示的路径。

In [ ]:
# ================================================================
# CELL 1: Global Configuration — Edit here only
# 全局配置 — 只需修改这里
# ================================================================

# ------ Input file paths / 输入文件路径 ------
# Reference voice: 20–30s clean vocal of the TARGET timbre you want
# 参考音色路径：20~30 秒的目标音色人声（你希望转换成谁的声音）
REF_PATH = "/kaggle/input/YOUR-DATASET-NAME/reference.wav"

# Source vocal: full-length song vocal to be converted
# 原始人声路径：需要被转换的完整歌曲人声
SRC_PATH = "/kaggle/input/YOUR-DATASET-NAME/source.wav"


# ------ Inference parameters / 推理参数 ------
# (These defaults work well for most Chinese songs. Tune only if output quality is poor.)
# （以下默认值对大多数中文歌曲效果良好，效果不满意时再调整。）

# Diffusion steps: higher = better quality but slower. Recommended: 40~60 for singing.
# 扩散步数：越高质量越好但越慢，歌声转换推荐 40~60
DIFFUSION_STEPS = 40

# Length adjustment: 1.0 = keep original duration. Increase slightly (e.g. 1.07) for smoother endings.
# 时长调整：1.0 = 不改变时长；微调到 1.07 可让尾音更自然
LENGTH_ADJUST = 1.0

# CFG guidance rate: higher = more like reference timbre, but too high causes noise.
# 引导强度：越高越贴近参考音色，太高（>0.8）会引入噪音，推荐 0.6~0.7
CFG_RATE = 0.7

# Auto F0 adjustment: False = keep original pitch/key, do not transpose.
# 自动音高校正：False = 保持原曲调性不做移调（推荐保持 False）
AUTO_F0 = False

# Semitone shift: 0 = no transposition. Use +/- integers to shift pitch.
# 半音偏移：0 = 不移调；正数升调，负数降调（通常保持 0）
SEMITONE_SHIFT = 0


# ------ Confirmation printout / 配置确认 ------
print("✅ Configuration loaded / 配置完成")
print(f"   Reference (参考音色): {REF_PATH}")
print(f"   Source    (原始人声): {SRC_PATH}")
print(f"   Diffusion steps: {DIFFUSION_STEPS}")
print(f"   CFG rate:        {CFG_RATE}")
print(f"   Length adjust:   {LENGTH_ADJUST}")
print(f"   Semitone shift:  {SEMITONE_SHIFT}")

## Cell 2 — Verify GPU / 检查 GPU 环境

Seed-VC requires a GPU. This cell confirms one is allocated.  
Seed-VC 必须使用 GPU 运行。本 Cell 确认 GPU 已正确分配。

**If you see ❌ / 如果显示 ❌：**  
Right panel → Settings → Accelerator → select `GPU T4 x2`  
右侧面板 → Settings → Accelerator → 选择 `GPU T4 x2`

In [ ]:
# ================================================================
# CELL 2: Verify GPU environment
# 检查 GPU 环境
# ================================================================
# Runs nvidia-smi to confirm a GPU is available.
# 通过 nvidia-smi 命令确认 GPU 已就绪。

import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)

if result.returncode == 0:
    print("✅ GPU is ready / GPU 已就绪：")
    # Print first 15 lines to avoid overly long output
    # 只打印前 15 行，避免输出过长
    lines = result.stdout.split("\n")
    for line in lines[:15]:
        print(line)
else:
    print("❌ No GPU detected / 未检测到 GPU")
    print("   → Right panel: Settings → Accelerator → T4 GPU")
    print("   → 右侧面板: Settings → Accelerator → T4 GPU")

## Cell 3 — Validate Input Audio Files / 检查输入音频文件

This cell checks that your audio files exist and are readable before anything else runs.  
本 Cell 在正式运行前，验证你的音频文件是否存在且可读。

**Common errors / 常见错误：**
- `❌ File not found` — check that your Kaggle Dataset is mounted (left Input panel) and `REF_PATH`/`SRC_PATH` in Cell 1 match exactly (case-sensitive, watch for spaces).  
  文件不存在 → 确认 Dataset 已挂载，Cell 1 中路径拼写完全一致（区分大小写，注意空格）。

In [ ]:
# ================================================================
# CELL 3: Validate input audio files
# 验证输入音频文件是否存在
# ================================================================
# Checks file existence and prints file size for both inputs.
# If a file is missing, lists directory contents to help diagnose the path.
# 检查两个输入文件是否存在并显示文件大小。
# 如果文件缺失，自动列出目录内容以帮助定位路径问题。

import os

print("🔍 Checking input files / 检查输入文件...\n")

errors = []
checks = [
    ("Reference / 参考音色", REF_PATH),
    ("Source    / 原始人声", SRC_PATH),
]

for label, path in checks:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"✅ {label}")
        print(f"   Path: {path}  ({size_mb:.1f} MB)")
    else:
        print(f"❌ {label}")
        print(f"   File not found / 文件不存在: {path}")
        errors.append(path)

if errors:
    print("\n⚠️  Troubleshooting / 排查建议:")
    print("   1. Is your Kaggle Dataset mounted? Check the left Input panel.")
    print("      Kaggle Dataset 是否已挂载？检查左侧 Input 面板。")
    print("   2. Does REF_PATH / SRC_PATH in Cell 1 exactly match your file path?")
    print("      Cell 1 中路径是否拼写完全正确（区分大小写，注意空格）？")
    print("   3. Directory listing for diagnosis / 目录内容诊断:")
    for path in errors:
        parent = os.path.dirname(path)
        if os.path.exists(parent):
            print(f"\n   Contents of {parent}:")
            for f in sorted(os.listdir(parent)):
                print(f"     - {f}")
        else:
            print(f"\n   ❌ Directory also missing / 目录不存在: {parent}")
else:
    print("\n✅ All files verified — ready to continue / 所有文件检查通过，可以继续")

## Cell 4 — Install Seed-VC / 安装 Seed-VC 及依赖

Clones the official Seed-VC repository and installs all Python dependencies.  
克隆 Seed-VC 官方仓库并安装所有 Python 依赖。

- **First run:** ~3–5 minutes / 首次运行约 3~5 分钟  
- **Subsequent runs:** skips clone if repo already exists / 仓库已存在时自动跳过克隆
- Model weights are downloaded automatically during inference — no manual download needed.  
  模型权重在推理时自动下载，无需手动准备。

> ℹ️ You may see pip dependency conflict warnings — these are harmless and can be ignored.  
> ℹ️ pip 可能输出依赖冲突警告，属于正常现象，可以忽略。

In [ ]:
# ================================================================
# CELL 4: Install Seed-VC and dependencies
# 安装 Seed-VC 及依赖
# ================================================================
# Clones the repo only if it doesn't already exist (safe to re-run).
# 仓库不存在时才克隆（可以安全重复运行）。

import os

REPO_DIR = "/kaggle/working/seed-vc"

# Step 1: Clone repository / 克隆仓库
if not os.path.exists(REPO_DIR):
    print("📥 Cloning Seed-VC repository / 克隆 Seed-VC 仓库...")
    !git clone https://github.com/Plachtaa/seed-vc.git {REPO_DIR}
    print("✅ Clone complete / 克隆完成")
else:
    print("✅ Repository already exists, skipping clone / 仓库已存在，跳过克隆")

# Step 2: Change to repo directory / 切换到仓库目录
%cd {REPO_DIR}

# Step 3: Install Python dependencies / 安装 Python 依赖
print("\n📦 Installing dependencies (~2–3 min) / 安装依赖包（约 2~3 分钟）...")
!pip install -q -r requirements.txt

print("\n✅ Environment ready / 环境安装完成")

## Cell 4.5 — Fix Protobuf Conflict (Kaggle only) / 修复 Protobuf 版本冲突（Kaggle 环境专用）

**Why this is needed / 为什么需要这步：**  
Kaggle pre-installs `protobuf 3.19.6`, but `transformers` requires `>=3.20`. This cell upgrades it.  
Kaggle 预装的 `protobuf 3.19.6` 版本过低，`transformers` 需要 `>=3.20`，本 Cell 进行升级修复。

**Expected output / 正常输出示例：**
```
protobuf version: 3.20.x
transformers version: 4.39.3
✅ Fix complete
```

> ⚠️ If you see a `NameError: pkg_resources is not defined` error, it's safe to ignore — the packages were installed correctly. Just continue to the next cell.  
> ⚠️ 如果出现 `NameError: pkg_resources is not defined`，可以忽略，包已正确安装，继续运行下一个 Cell 即可。

In [ ]:
# ================================================================
# CELL 4.5: Fix protobuf version conflict (Kaggle environment only)
# 修复 protobuf 版本冲突（Kaggle 环境专用）
# ================================================================
# Root cause: Kaggle ships protobuf 3.19.6; transformers needs >=3.20.
# 根本原因：Kaggle 预装 protobuf 3.19.6，transformers 需要 >=3.20。

# Upgrade protobuf and pin transformers to a compatible version
# 升级 protobuf 并锁定兼容版本的 transformers
!pip install -q "protobuf>=3.20.0,<4.0.0"
!pip install -q "transformers==4.39.3"

# Verify installed versions / 验证安装版本
import pkg_resources
print("protobuf version  :", pkg_resources.get_distribution("protobuf").version)
print("transformers version:", pkg_resources.get_distribution("transformers").version)
print("✅ Fix complete / 修复完成，继续运行下一个 Cell")

## Cell 5 — Prepare Working Directory / 准备工作目录，复制音频

Copies your input audio files from the read-only Kaggle Dataset mount into the writable working directory.  
将你的输入音频从只读的 Kaggle Dataset 挂载点，复制到可写的工作目录。

**Output paths after this cell / 本 Cell 完成后的路径：**
```
/kaggle/working/test_audio/reference.wav   ← 参考音色
/kaggle/working/test_audio/source.wav      ← 原始人声
```

> 💡 If you want to re-run inference with fresh output, see Cell 5.1 to clear previous results.  
> 💡 如果需要清空之前的输出重新推理，见 Cell 5.1。

In [ ]:
# ================================================================
# CELL 5: Prepare working directory and copy audio files
# 准备工作目录，复制音频文件
# ================================================================
# Kaggle Dataset mounts are read-only. We copy files to a writable
# working directory before inference.
# Kaggle Dataset 挂载为只读，需要先复制到可写的工作目录才能推理。

import shutil, os

AUDIO_DIR  = "/kaggle/working/test_audio"
OUTPUT_DIR = "/kaggle/working/output"

# Create directories if they don't exist / 创建目录（已存在则忽略）
os.makedirs(AUDIO_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copy input files to working directory / 复制音频到工作目录
ref_dst = f"{AUDIO_DIR}/reference.wav"
src_dst = f"{AUDIO_DIR}/source.wav"

print("📋 Copying audio files / 复制音频文件...")
shutil.copy2(REF_PATH, ref_dst)
print(f"   ✅ reference.wav  ← {REF_PATH}")
shutil.copy2(SRC_PATH, src_dst)
print(f"   ✅ source.wav     ← {SRC_PATH}")

# Confirm file sizes / 确认文件大小
print("\n📁 Working directory contents / 工作目录文件：")
for f in sorted(os.listdir(AUDIO_DIR)):
    size_mb = os.path.getsize(f"{AUDIO_DIR}/{f}") / 1024 / 1024
    print(f"   {f}  ({size_mb:.1f} MB)")

print("\n✅ Ready for inference / 准备就绪，可以开始推理")

### Cell 5.1 — Clear previous test outputs (optional) / 清空之前的测试输出（可选）

Run this **only** if you want to delete previous test results and start fresh.  
**仅在**想清空之前的测试输出、重新开始时才运行这个 Cell，否则跳过。

In [ ]:
# ================================================================
# CELL 5.1: Clear previous test outputs (optional — skip if not needed)
# 清空之前的测试输出（可选，不需要时跳过）
# ================================================================
# Deletes .wav files in the test output directory.
# 删除测试输出目录中的 .wav 文件。

import glob, os

files = glob.glob("/kaggle/working/output_test/*.wav")
for f in files:
    os.remove(f)

print(f"🗑️  Removed {len(files)} file(s) / 已删除 {len(files)} 个文件")
print("✅ Test output directory cleared / 测试输出目录已清空")

## Cell 6 — Test Inference on 20s Preview / 20 秒片段测试推理

**Run this before the full song (Cell 7)** to verify the setup works and preview the effect.  
**在跑完整歌曲之前先运行此 Cell**，验证环境正常并预听转换效果。

**What this cell does / 本 Cell 的工作：**
1. Automatically finds the loudest 20-second segment in `source.wav` (highest amplitude = most vocal activity).  
   自动找出 `source.wav` 中振幅最大的 20 秒片段（振幅最高 = 人声最活跃的部分）。
2. Runs Seed-VC inference on that 20s clip.  
   对该 20 秒片段执行 Seed-VC 推理。
3. Applies loudness normalization via ffmpeg.  
   用 ffmpeg 进行响度归一化后处理。

**Expected runtime / 预计耗时：** ~1–3 minutes on T4 GPU / T4 GPU 上约 1~3 分钟

> ⚠️ If the output is silent or distorted, run the diagnostic cells (6.1 ~ 6.3) below before proceeding.  
> ⚠️ 如果输出无声或失真，请先运行下方诊断 Cell（6.1 ~ 6.3）再继续。

In [ ]:
# ================================================================
# CELL 6: Test inference on a 20-second preview clip
# 对 20 秒预览片段进行测试推理
# ================================================================

import os
import numpy as np
import soundfile as sf
import glob

# ------ Inference parameters for test / 测试推理参数 ------
# These are tuned for smooth, expressive Chinese vocals with vibrato.
# 针对国风抒情、颤音细腻、回转丝滑的效果调优。
DIFFUSION_STEPS = 60   # Higher = better quality, slower / 越高质量越好但越慢
LENGTH_ADJUST   = 1.08 # Slight stretch for smoother phrase endings / 微拉伸使尾音更自然
CFG_RATE        = 0.6  # Guidance strength (0.6 avoids harsh metallic artifacts) / 引导强度
AUTO_F0         = False  # Keep original pitch — do not transpose / 保持原调，不移调
SEMITONE_SHIFT  = 0      # No pitch shift / 不移调

# ------ Setup paths / 路径设置 ------
os.makedirs("/kaggle/working/output_test", exist_ok=True)
src_path     = "/kaggle/working/test_audio/source.wav"
preview_path = "/kaggle/working/test_audio/source_preview.wav"

# ------ Extract the best 20-second segment / 自动提取最优 20 秒片段 ------
# Strategy: slide a 20s window across the file and pick the segment with
# the highest peak amplitude (most vocal energy).
# 策略：用 20 秒滑动窗口遍历文件，选取峰值振幅最高的片段（人声能量最强）。
data, sr = sf.read(src_path)
mono = data[:, 0]            # Use left channel / 使用左声道
total_samples = len(mono)

window_size = int(20 * sr)   # 20 seconds in samples / 20 秒对应的采样点数
step_size   = int(5 * sr)    # Slide in 5-second steps / 每次滑动 5 秒
best_amp    = 0
best_start  = 0

for start in range(0, total_samples - window_size + 1, step_size):
    segment = mono[start : start + window_size]
    current_amp = np.max(np.abs(segment))
    if current_amp > best_amp:
        best_amp   = current_amp
        best_start = start

best_audio = mono[best_start : best_start + window_size]
sf.write(preview_path, best_audio, sr, subtype="PCM_16")

start_sec = best_start / sr
print(f"✅ Preview clip extracted / 测试片段已提取")
print(f"   Segment start: {start_sec:.1f}s | Peak amplitude: {best_amp:.4f}")
print(f"   Saved to: {preview_path}")

# ------ Run inference / 执行推理 ------
print("\n⏳ Running test inference / 开始测试推理...")
%cd /kaggle/working/seed-vc

!python inference.py \
    --source "{preview_path}" \
    --target "/kaggle/working/test_audio/reference.wav" \
    --output "/kaggle/working/output_test" \
    --diffusion-steps {DIFFUSION_STEPS} \
    --length-adjust {LENGTH_ADJUST} \
    --inference-cfg-rate {CFG_RATE} \
    --f0-condition True \
    --auto-f0-adjust {AUTO_F0} \
    --semi-tone-shift {SEMITONE_SHIFT} \
    --fp16 False

# ------ Post-processing: loudness normalization / 后处理：响度归一化 ------
# loudnorm: normalize to -16 LUFS with max LRA 12 LU
# afftdn: frequency-domain noise reduction
# loudnorm: 响度归一化至 -16 LUFS；afftdn: 频域降噪
for f in glob.glob("/kaggle/working/output_test/*.wav"):
    if "_TUNE" not in f:  # Avoid reprocessing / 避免重复处理
        final = f.replace(".wav", "_TUNE.wav")
        !ffmpeg -y -i "{f}" -filter:a "loudnorm=I=-16:LRA=12,afftdn" "{final}" -loglevel error

print("\n🎉 Test complete / 测试完成！")
print("   Check /kaggle/working/output_test/ for _TUNE.wav files")
print("   查看 /kaggle/working/output_test/ 目录中的 _TUNE.wav 文件")

---
### 🔧 Diagnostic Tools / 诊断工具（仅出错时使用）

Run **only if** the test output in Cell 6 is silent or has issues.  
**仅当** Cell 6 输出无声或有问题时才运行以下诊断 Cell。

**Normal reference values / 正常参考数据：**
```
Total samples: ~10,000,000+
Peak amplitude: > 0.5  (if < 0.1, the audio may be silent or wrong channel)
```

---
#### Cell 6.1 — Check amplitude of last 20 seconds / 检查最后 20 秒的振幅

In [ ]:
# ================================================================
# CELL 6.1 [DIAGNOSTIC]: Check amplitude of last 20 seconds
# [诊断] 检查最后 20 秒的振幅
# ================================================================
# Use when: output audio is silent — verifies source file has vocal content.
# 当输出无声时使用：验证原始文件中确实有人声。
# Normal: peak amplitude > 0.5 / 正常值：峰值振幅 > 0.5

import numpy as np
import soundfile as sf

src = "/kaggle/working/test_audio/source.wav"
data, sr = sf.read(src)

mono = data[:, 0]   # Left channel / 左声道
total_samples = len(mono)

print("Source file info / 源文件信息：")
print(f"  Sample rate    / 采样率:  {sr} Hz")
print(f"  Total samples  / 总采样点: {total_samples}")
print(f"  Peak amplitude / 最大振幅: {np.max(np.abs(mono)):.6f}")
print(f"  Duration       / 时长:     {total_samples / sr:.1f}s")

# Check last 20 seconds / 检查最后 20 秒
last_20s = mono[max(0, total_samples - 20 * sr):]
amp_last = np.max(np.abs(last_20s))
print(f"\nLast 20s peak amplitude / 最后 20 秒峰值振幅: {amp_last:.6f}")

# Save for manual listening / 保存供试听
sf.write("/kaggle/working/test_audio/LAST_20S_TEST.wav", last_20s, sr, subtype="PCM_16")
print("Saved: /kaggle/working/test_audio/LAST_20S_TEST.wav")

if amp_last < 0.1:
    print("\n⚠️  Amplitude very low — source file may not contain vocal audio.")
    print("    振幅极低 — 原始文件可能不包含人声音频。")
else:
    print("\n✅ Amplitude looks normal / 振幅正常")

#### Cell 6.2 — Inspect preview clip / 检查预览片段的音频数据

In [ ]:
# ================================================================
# CELL 6.2 [DIAGNOSTIC]: Inspect the 20s preview clip
# [诊断] 检查 20 秒预览片段的音频数据
# ================================================================
# Use when: unsure whether the extracted preview clip actually has vocal.
# 当不确定提取的预览片段是否包含人声时使用。
# Normal: peak amplitude > 0.5 / 正常值：峰值振幅 > 0.5

import soundfile as sf
import numpy as np

x, sr = sf.read("/kaggle/working/test_audio/source_preview.wav")

print("Preview clip stats / 预览片段统计：")
print(f"  Sample rate    / 采样率:    {sr} Hz")
print(f"  Total samples  / 数据长度:  {len(x)}")
print(f"  Peak amplitude / 最大振幅:  {np.max(np.abs(x)):.6f}")
print(f"  First 10 samples / 前 10 个采样点: {x[:10]}")

if np.max(np.abs(x)) < 0.1:
    print("\n⚠️  Very low amplitude — check Cell 6.3 for channel diagnosis.")
    print("    振幅极低 — 请运行 Cell 6.3 检查声道问题。")
else:
    print("\n✅ Preview clip has vocal content / 预览片段包含人声")

#### Cell 6.3 — Left/right channel separation test / 左右声道分离测试

Use when amplitude is normal but output is still silent — one channel may be empty.  
当振幅正常但输出仍无声时使用 — 可能有一个声道为空。

In [ ]:
# ================================================================
# CELL 6.3 [DIAGNOSTIC]: Left/right channel separation test
# [诊断] 左右声道分离测试
# ================================================================
# Use when: output is silent despite normal peak amplitude.
# Separates channels and saves each independently to identify which has vocal.
# 当振幅正常但输出仍无声时使用。
# 分别保存左声道、右声道和混合版本，帮助判断哪个声道有人声。

import numpy as np
import soundfile as sf

src_path  = "/kaggle/working/test_audio/source.wav"
data, sr  = sf.read(src_path)

print("Audio file info / 音频文件信息：")
print(f"  Sample rate / 采样率: {sr} Hz")
print(f"  Shape / 形状: {data.shape}  (samples × channels)")
print(f"  Channels / 声道数: {data.shape[1]}")

# Analyze each channel / 分析各声道
left  = data[:, 0]
right = data[:, 1]
mix   = (left + right) / 2

amp_left  = np.max(np.abs(left))
amp_right = np.max(np.abs(right))
amp_mix   = np.max(np.abs(mix))

print(f"\nChannel amplitudes / 各声道振幅：")
print(f"  Left  / 左声道: {amp_left:.6f}")
print(f"  Right / 右声道: {amp_right:.6f}")
print(f"  Mix   / 混合:   {amp_mix:.6f}")

# Diagnosis / 诊断结论
print("\n👉 Diagnosis / 诊断结论：")
if amp_left > 0.1:  print("  ✅ Left channel has vocal / 左声道有人声")
if amp_right > 0.1: print("  ✅ Right channel has vocal / 右声道有人声")
if amp_left < 0.01 and amp_right < 0.01:
    print("  ❌ Both channels are silent — file may be corrupted.")
    print("     两个声道都是空的 — 文件本身可能有问题！")

# Save all three versions for manual inspection / 保存三个版本供手动试听
sf.write("/kaggle/working/test_audio/only_left.wav",  left,  sr, subtype="PCM_16")
sf.write("/kaggle/working/test_audio/only_right.wav", right, sr, subtype="PCM_16")
sf.write("/kaggle/working/test_audio/mix_stereo.wav", mix,   sr, subtype="PCM_16")

print("\n✅ Saved for manual listening / 已保存供试听：")
print("   /kaggle/working/test_audio/only_left.wav")
print("   /kaggle/working/test_audio/only_right.wav")
print("   /kaggle/working/test_audio/mix_stereo.wav")

# Suggestion / 建议
if amp_left < 0.01 and amp_right > 0.1:
    print("\n💡 Right channel has vocal — change Cell 6 to use data[:, 1] instead.")
    print("   右声道有人声 — 在 Cell 6 中将 data[:, 0] 改为 data[:, 1] 再试。")

---
## Cell 7 — Full Song Inference / 完整歌曲推理

**Run this only after Cell 6 test passes without issues.**  
**只有 Cell 6 测试正常后才运行此 Cell。**

**Expected runtime / 预计耗时：** ~15–30 minutes for a 3–5 min song on T4 / T4 上 3~5 分钟歌曲约需 15~30 分钟

**Output / 输出：**  
`/kaggle/working/output_full/*_FINAL.wav` — post-processed final file  
`/kaggle/working/output_full/*_FINAL.wav` — 经后处理的最终文件

**Tuning reference / 调参参考：**

| Issue / 问题 | Action / 解决 | Parameter / 参数 |
|---|---|---|
| Not enough like reference / 音色不像参考 | Increase guidance / 提高引导强度 | `CFG_RATE = 0.8` |
| Too noisy/harsh / 噪音大或过于夸张 | Decrease guidance / 降低引导强度 | `CFG_RATE = 0.5` |
| Pitch too high/low / 音调偏高或偏低 | Adjust semitones / 调整半音 | `SEMITONE_SHIFT = ±1~3` |
| Output duration changed / 时长变了 | Lock duration / 锁定时长 | `LENGTH_ADJUST = 1.0` |
| Blurry/unclear output / 输出模糊 | More diffusion steps / 增加扩散步数 | `DIFFUSION_STEPS = 60` |
| Inference too slow / 推理太慢 | Fewer diffusion steps / 减少扩散步数 | `DIFFUSION_STEPS = 25` |

In [ ]:
# ================================================================
# CELL 7: Full song inference
# 完整歌曲推理
# ================================================================

import os, glob
import soundfile as sf
import numpy as np

# ------ Inference parameters / 推理参数 ------
# Same tuning as Cell 6 test — change here to adjust the full song output.
# 与 Cell 6 测试一致；如需调整完整版效果在此修改。
DIFFUSION_STEPS = 60    # Diffusion steps / 扩散步数 (quality vs speed)
LENGTH_ADJUST   = 1.07  # Duration multiplier / 时长调整系数
CFG_RATE        = 0.6   # Guidance rate / 引导强度
AUTO_F0         = False  # Auto pitch correction / 自动音高校正 (keep False)
SEMITONE_SHIFT  = 0      # Pitch shift in semitones / 半音偏移

# ------ Paths / 路径 ------
SOURCE_WAV    = "/kaggle/working/test_audio/source.wav"
REFERENCE_WAV = "/kaggle/working/test_audio/reference.wav"
OUTPUT_DIR    = "/kaggle/working/output_full"

os.makedirs(OUTPUT_DIR, exist_ok=True)
%cd /kaggle/working/seed-vc

print("⏳ Starting full song inference / 开始完整歌曲推理 (~15–30 min)...")
print(f"   Source:    {SOURCE_WAV}")
print(f"   Reference: {REFERENCE_WAV}")
print(f"   Output:    {OUTPUT_DIR}")

# ------ Run inference / 执行推理 ------
!python inference.py \
    --source  "{SOURCE_WAV}" \
    --target  "{REFERENCE_WAV}" \
    --output  "{OUTPUT_DIR}" \
    --diffusion-steps {DIFFUSION_STEPS} \
    --length-adjust   {LENGTH_ADJUST} \
    --inference-cfg-rate {CFG_RATE} \
    --f0-condition True \
    --auto-f0-adjust False \
    --semi-tone-shift {SEMITONE_SHIFT} \
    --fp16 False

# ------ Post-processing: loudness normalization / 后处理：响度归一化 ------
# loudnorm: EBU R128 normalization to -16 LUFS
# afftdn: frequency-domain noise reduction
# loudnorm: EBU R128 响度归一化至 -16 LUFS；afftdn: 频域降噪
print("\n🔧 Post-processing / 后处理中...")
for f in glob.glob(f"{OUTPUT_DIR}/*.wav"):
    if "_FINAL" not in f:  # Avoid reprocessing / 避免重复处理
        final = f.replace(".wav", "_FINAL.wav")
        !ffmpeg -y -i "{f}" -filter:a "loudnorm=I=-16:LRA=12,afftdn" "{final}" -loglevel error

# ------ Summary / 结果汇总 ------
print("\n✅ Inference complete / 推理完成！")
print("\nOutput files / 输出文件：")
for f in sorted(glob.glob(f"{OUTPUT_DIR}/*_FINAL.wav")):
    data, sr = sf.read(f)
    duration = len(data) / sr
    peak_amp = np.max(np.abs(data))
    print(f"   {os.path.basename(f)}")
    print(f"     Duration / 时长: {duration:.1f}s  |  Peak amplitude / 峰值振幅: {peak_amp:.4f}")

print("\n📥 Run Cell 8 to package results for download / 运行 Cell 8 打包下载")

## Cell 8 — Package and Download Results / 打包下载结果

Packages all output `.wav` files into a timestamped `.zip` for easy download from the Kaggle Output panel.  
将所有输出的 `.wav` 文件打包成带时间戳的 `.zip`，方便从 Kaggle Output 面板一键下载。

**How to download / 如何下载：**  
Right panel → Output tab → find `seedvc_result_YYYYMMDD_HHMM.zip` → Download  
右侧面板 → Output 标签页 → 找到 `seedvc_result_YYYYMMDD_HHMM.zip` → 下载

In [ ]:
# ================================================================
# CELL 8: Package output files for download
# 打包输出文件以供下载
# ================================================================
# Collects all .wav files from test and full-song output directories,
# compresses them into a single .zip with a timestamp in the filename.
# 收集测试和完整推理目录中的所有 .wav 文件，
# 打包成带时间戳的单个 .zip 文件。

import os, zipfile, glob
from datetime import datetime

# Timestamped output zip / 带时间戳的输出包
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_path  = f"/kaggle/working/seedvc_result_{timestamp}.zip"

# Collect all output wav files / 收集所有输出 wav 文件
output_dirs = [
    "/kaggle/working/output_test",  # 20s test results / 测试结果
    "/kaggle/working/output_full",  # Full song results / 完整歌曲结果
    "/kaggle/working/output",       # Legacy output dir / 旧版输出目录
]

files_to_pack = []
for d in output_dirs:
    if os.path.exists(d):
        for f in glob.glob(f"{d}/*.wav"):
            files_to_pack.append((f, d))

if files_to_pack:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fp, source_dir in files_to_pack:
            # Prefix to distinguish test vs full / 加前缀区分测试和完整版
            if "output_test" in source_dir:
                arcname = "TEST_"  + os.path.basename(fp)
            elif "output_full" in source_dir:
                arcname = "FULL_"  + os.path.basename(fp)
            else:
                arcname = os.path.basename(fp)
            zf.write(fp, arcname)

    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f"✅ Package created / 打包完成")
    print(f"   File: {zip_path}")
    print(f"   Size: {size_mb:.1f} MB")
    print(f"\n   Contents / 包含文件：")
    for fp, _ in files_to_pack:
        print(f"   - {os.path.basename(fp)}")
    print("\n📥 Download from the right panel → Output tab")
    print("   右侧面板 → Output 标签页下载")
else:
    print("❌ No output .wav files found / 未找到输出 wav 文件")
    print("   → Run Cell 6 (test) or Cell 7 (full song) first")
    print("   → 请先运行 Cell 6（测试）或 Cell 7（完整歌曲）")

---

## 📌 Notes & Reminders / 注意事项

- **Seed-VC converts timbre, not singing style.**  
  The output preserves the melody and phrasing of `source.wav`, with the voice characteristics of `reference.wav`.  
  **Seed-VC 转换的是音色，不是唱法。**  
  输出保留 `source.wav` 的旋律和唱法，换上 `reference.wav` 的声音特征。

- **Both audio files must be clean vocals** (no background music).  
  Use [UVR5](https://github.com/Anjok07/ultimatevocalremovergui) or [Demucs](https://github.com/facebookresearch/demucs) to separate tracks first.  
  **两个音频都必须是纯人声**（无背景音乐）。  
  请提前用 UVR5 或 Demucs 拆轨，否则背景音乐会严重影响转换效果。

- **Kaggle free GPU has weekly usage limits.** Always test with Cell 6 before running the full song.  
  **Kaggle 免费 GPU 有每周时长限制。** 务必先用 Cell 6 测试通过，再运行完整推理。

- Model weights are downloaded automatically on first inference (~1–2 GB, cached for the session).  
  模型权重在首次推理时自动下载（约 1~2 GB，本次会话内缓存）。

---

## 🔗 References / 参考链接

- [Seed-VC GitHub](https://github.com/Plachtaa/seed-vc) — Official repository / 官方仓库  
- [UVR5](https://github.com/Anjok07/ultimatevocalremovergui) — Vocal separation / 人声拆轨工具  
- [Demucs](https://github.com/facebookresearch/demucs) — Source separation / 音源分离  
- [AI Assistant: ]Claude(https://claude.ai) by Anthropic / 协助文档编写与代码整理